# IEEE-CIS Fraud Detection — Logistic Regression

Sections: **Cleaning → Feature Engineering → Feature Selection → Training**



In [1]:
!pip install dagshub mlflow scikit-learn pandas numpy -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/

In [2]:
import os, gc, warnings
import numpy as np
import pandas as pd
import mlflow, mlflow.sklearn
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.impute import SimpleImputer
from mlflow.models.signature import infer_signature
from scipy.stats import loguniform, uniform

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['MLFLOW_TRACKING_USERNAME'] = 'dgrig23'
os.environ['MLFLOW_TRACKING_PASSWORD'] = secrets.get_secret('DAGSHUB_TOKEN')

REPO = 'dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning'
mlflow.set_tracking_uri(f'https://dagshub.com/{REPO}.mlflow')
mlflow.set_experiment('LogisticRegression_Training')
EXP_PREFIX = 'LR'
BASE = '/kaggle/input/competitions/ieee-fraud-detection/'
print('Environment Ready.')

Environment Ready.


## 1. Cleaning

In [3]:
print('Loading data...')
train_trx = pd.read_csv(BASE + 'train_transaction.csv')
train_idn = pd.read_csv(BASE + 'train_identity.csv')

train_idn.columns = train_idn.columns.str.replace('-', '_')
train = train_trx.merge(train_idn, on='TransactionID', how='left')
del train_trx, train_idn
gc.collect()

train = reduce_mem_usage(train)
print(f'Compressed Train shape: {train.shape}')

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Cleaning'):
    HIGH_MISS = 0.5
    miss = train.isnull().mean()
    high_miss_cols = miss[miss > HIGH_MISS].index.tolist()
    
    train.drop(columns=high_miss_cols + ['TransactionID'], inplace=True, errors='ignore')
    y = train.pop('isFraud').copy()
    fraud_rate = y.mean()
    
    mlflow.log_params({
        'high_miss_threshold': HIGH_MISS,
        'cols_dropped': len(high_miss_cols),
        'class_weight': 'balanced',
        'note': 'stricter_miss_thresh_for_LR'
    })
    mlflow.log_metrics({
        'fraud_rate': round(float(fraud_rate), 4),
        'cols_after': train.shape[1]
    })
    print(f'Fraud rate: {fraud_rate:.4f} | Columns remaining: {train.shape[1]}')

Loading data...
Compressed Train shape: (590540, 434)
Fraud rate: 0.0350 | Columns remaining: 218
🏃 View run LR_Cleaning at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/e35c3a0341c64245a2dc079c2a354ac3
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2


## 2. Feature Engineering

In [4]:
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Engineering'):
    cols_before = train.shape[1]
    
    if 'TransactionDT' in train.columns:
        train['hour'] = ((train['TransactionDT']/3600)%24).astype(np.float32)
        train['dayofweek'] = ((train['TransactionDT']/(3600*24))%7).astype(np.float32)
        train['hour_sin'] = np.sin(2*np.pi*train['hour']/24).astype(np.float32)
        train['hour_cos'] = np.cos(2*np.pi*train['hour']/24).astype(np.float32)
        train['dow_sin'] = np.sin(2*np.pi*train['dayofweek']/7).astype(np.float32)
        train['dow_cos'] = np.cos(2*np.pi*train['dayofweek']/7).astype(np.float32)
        train.drop(columns=['TransactionDT'], inplace=True)
        
    for col in ['P_emaildomain','R_emaildomain']:
        if col in train.columns:
            train[col+'_suffix'] = train[col].str.split('.').str[-1].fillna('unknown')
            train[col+'_domain'] = train[col].str.split('.').str[0].fillna('unknown')
            
    if 'P_emaildomain' in train.columns and 'R_emaildomain' in train.columns:
        train['email_match'] = (train['P_emaildomain'] == train['R_emaildomain']).astype(np.int8)
        
    if 'TransactionAmt' in train.columns:
        train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
        train['TransactionAmt_cents'] = (train['TransactionAmt']%1).astype(np.float32)
        train['amt_is_round'] = (train['TransactionAmt']%1 == 0).astype(np.int8)
        p99 = train['TransactionAmt'].quantile(0.99)
        train['TransactionAmt_capped'] = train['TransactionAmt'].clip(upper=p99)
        
    for g in ['card1','addr1']:
        if g in train.columns:
            train[f'{g}_count'] = train.groupby(g)[g].transform('count')
            train[f'{g}_count_log'] = np.log1p(train[f'{g}_count'])
            
    mlflow.log_params({
        'time': 'hour,dow+cyclic_sincos',
        'email': 'suffix,domain,match',
        'amount': 'log,cents,round,capped_p99',
        'aggs': 'card1_count,addr1_count',
        'cyclic_encoding': True
    })
    mlflow.log_metrics({
        'new_features': train.shape[1] - cols_before,
        'total_features': train.shape[1]
    })
    print(f'Engineering Complete. Features: {cols_before} → {train.shape[1]}')

Engineering Complete. Features: 218 → 233
🏃 View run LR_Feature_Engineering at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/8e01f438b5da4f99989de1e2d2d4caf5
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2


## 3. Feature Selection

In [5]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
le_store = {}
for col in cat_cols:
    le = LabelEncoder()
    train[col] = train[col].fillna('unknown').astype(str)
    le.fit(list(train[col].unique()) + ['unknown'])
    le_store[col] = le
    train[col] = le.transform(train[col])

num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
train[num_cols] = imputer.fit_transform(train[num_cols])
train[num_cols] = scaler.fit_transform(train[num_cols])

X_fs, _, y_fs, _ = train_test_split(train, y, train_size=0.20, stratify=y, random_state=42)
X_tr_fs, X_va_fs, y_tr_fs, y_va_fs = train_test_split(X_fs, y_fs, test_size=0.20, stratify=y_fs, random_state=42)

print(f"Sampling complete. Selection training size: {X_tr_fs.shape}")

Sampling complete. Selection training size: (94486, 233)


In [6]:
def quick_eval(features):
    m = LogisticRegression(C=1.0, max_iter=150, tol=1e-2, class_weight='balanced', solver='saga', n_jobs=-1, random_state=42)
    m.fit(X_tr_fs[features], y_tr_fs)
    return roc_auc_score(y_va_fs, m.predict_proba(X_va_fs[features])[:,1])

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Feature_Selection'):
    all_features = train.columns.tolist()
    auc_all = quick_eval(all_features)
    print(f'Strategy A – All ({len(all_features)}): AUC = {auc_all:.5f}')

    vt = VarianceThreshold(threshold=0.05)
    vt.fit(train)
    vt_features = train.columns[vt.get_support()].tolist()
    auc_vt = quick_eval(vt_features)
    print(f'Strategy B – VarianceThreshold ({len(vt_features)}): AUC = {auc_vt:.5f}')

    corr = train.corrwith(y).abs()
    corr_features = corr[corr >= 0.02].index.tolist()
    auc_corr = quick_eval(corr_features)
    print(f'Strategy C – Correlation ({len(corr_features)}): AUC = {auc_corr:.5f}')

    lr_l1 = LogisticRegression(C=0.1, penalty='l1', solver='saga', max_iter=150, tol=1e-2, class_weight='balanced', random_state=42, n_jobs=-1)
    sfm = SelectFromModel(lr_l1, threshold='mean')
    sfm.fit(X_tr_fs, y_tr_fs)
    l1_features = train.columns[sfm.get_support()].tolist()
    auc_l1 = quick_eval(l1_features)
    print(f'Strategy D – L1 Lasso ({len(l1_features)}): AUC = {auc_l1:.5f}')

    best_strategy, best_auc, final_features = max(
        [('all', auc_all, all_features), ('variance_threshold', auc_vt, vt_features),
         ('correlation', auc_corr, corr_features), ('l1_lasso', auc_l1, l1_features)],
        key=lambda x: x[1]
    )
    
    mlflow.log_params({'selected': best_strategy, 'n_final': len(final_features), 'sampling_rate': 0.20})
    mlflow.log_metrics({'best_auc': round(best_auc, 5)})
    print(f'→ Best: {best_strategy} | AUC: {best_auc:.5f}')

Strategy A – All (233): AUC = 0.82812
Strategy B – VarianceThreshold (233): AUC = 0.82812
Strategy C – Correlation (147): AUC = 0.81923
Strategy D – L1 Lasso (86): AUC = 0.83003
→ Best: l1_lasso | AUC: 0.83003
🏃 View run LR_Feature_Selection at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/76ae9aaf663a4212a7460059334e3a53
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2


## 4. Training — Logistic Regression

In [7]:
X = train[final_features].copy()
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# (a) Underfitting 
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Underfit_Config'):
    m_under = LogisticRegression(C=1e-9, penalty='l2', solver='saga', max_iter=1, 
                                 tol=1e-1, class_weight='balanced', random_state=42, n_jobs=-1)
    m_under.fit(X_tr, y_tr)
    
    tr_auc = roc_auc_score(y_tr, m_under.predict_proba(X_tr)[:, 1])
    va_auc = roc_auc_score(y_va, m_under.predict_proba(X_va)[:, 1])
    
    mlflow.log_params({'C': 1e-9, 'max_iter': 1, 'note': 'forced_underfit'})
    mlflow.log_metrics({'train_auc': round(tr_auc, 5), 'val_auc': round(va_auc, 5), 'overfit_gap': round(tr_auc-va_auc, 5)})
    print(f'Underfit — Train: {tr_auc:.5f} | Val: {va_auc:.5f}')

# (b) Overfitting
with mlflow.start_run(run_name=f'{EXP_PREFIX}_Overfit_Config'):
    X_tiny = X_tr[:500]
    y_tiny = y_tr[:500]
    
    m_over = LogisticRegression(C=1e9, penalty='l2', solver='liblinear', max_iter=1000, 
                                class_weight='balanced', random_state=42)
    m_over.fit(X_tiny, y_tiny)
    
    tr_auc = roc_auc_score(y_tiny, m_over.predict_proba(X_tiny)[:, 1])
    va_auc = roc_auc_score(y_va, m_over.predict_proba(X_va)[:, 1])
    
    mlflow.log_params({'C': 1e9, 'train_size': 500, 'note': 'forced_overfit_via_tiny_sample'})
    mlflow.log_metrics({'train_auc': round(tr_auc, 5), 'val_auc': round(va_auc, 5), 'overfit_gap': round(tr_auc-va_auc, 5)})
    print(f'Overfit  — Train: {tr_auc:.5f} | Val: {va_auc:.5f} | Gap: {tr_auc-va_auc:.5f}')

Underfit — Train: 0.75923 | Val: 0.76287
🏃 View run LR_Underfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/9fdcaaa96cb347d1a8acdc3aede38734
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2
Overfit  — Train: 1.00000 | Val: 0.55569 | Gap: 0.44431
🏃 View run LR_Overfit_Config at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/d0672a83792c447fbd1431ed303cb042
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2


In [8]:
# (c) RandomizedSearch
with mlflow.start_run(run_name=f'{EXP_PREFIX}_RandomizedSearch') as run_rs:
    X_rs_sample, _, y_rs_sample, _ = train_test_split(
        train[final_features], y, train_size=0.25, stratify=y, random_state=42
    )

    param_grid = [
        {
            'C': loguniform(1e-2, 1e2),
            'penalty': ['l1', 'l2'],
            'solver': ['saga'],
            'max_iter': [100] 
        },
        {
            'C': loguniform(1e-2, 1e2),
            'penalty': ['elasticnet'],
            'solver': ['saga'],
            'l1_ratio': uniform(0, 1),
            'max_iter': [100]
        }
    ]
    
    base_clf = LogisticRegression(class_weight='balanced', tol=1e-1, random_state=42, n_jobs=-1)
    
    skf5 = StratifiedKFold(5, shuffle=True, random_state=42)
    
    rs = RandomizedSearchCV(
        base_clf, 
        param_distributions=param_grid, 
        n_iter=10, 
        cv=skf5, 
        scoring='roc_auc',
        n_jobs=2, 
        return_train_score=True, 
        random_state=42,
        verbose=1, 
        error_score=0.0
    )
    
    print(f"Starting 5-Fold Search on {len(X_rs_sample)} rows (25% sample)...")
    rs.fit(X_rs_sample, y_rs_sample)

    for i, params in enumerate(rs.cv_results_['params']):
        p_name = params['penalty']
        label = f"C{round(params['C'],3)}_{p_name}_iter{params['max_iter']}"
        with mlflow.start_run(run_name=f'LR_RS_{label}', nested=True):
            mlflow.log_params({str(k): str(v) for k,v in params.items()})
            mlflow.log_metrics({
                'cv_auc_mean': round(float(rs.cv_results_['mean_test_score'][i]), 5),
                'cv_auc_std': round(float(rs.cv_results_['std_test_score'][i]), 5),
                'train_auc_mean': round(float(rs.cv_results_['mean_train_score'][i]), 5),
                'overfit_gap': round(float(rs.cv_results_['mean_train_score'][i] - rs.cv_results_['mean_test_score'][i]), 5),
            })

    best_params = rs.best_params_
    mlflow.log_params({'best_'+k: str(v) for k,v in best_params.items()})
    mlflow.log_metric('best_cv_auc', round(rs.best_score_, 5))
    print(f'Best CV AUC: {rs.best_score_:.5f} | Params: {best_params}')

Starting 5-Fold Search on 147635 rows (25% sample)...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
🏃 View run LR_RS_C15.352_l1_iter100 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/56c22403ce2a4523831b01eb5c9c084c
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2
🏃 View run LR_RS_C13.145_l1_iter100 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/0a742da68021446ca4c27deece75ea0e
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2
🏃 View run LR_RS_C0.607_l1_iter100 at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/eff639fece1840d2ac1df7d172059718
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2
🏃 View run LR_RS

In [9]:
import pandas as pd
import numpy as np
import gc
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

print('Reloading raw data for final pipeline...')
raw_trx = pd.read_csv(BASE + 'train_transaction.csv')
raw_idn = pd.read_csv(BASE + 'train_identity.csv')
raw_idn.columns = raw_idn.columns.str.replace('-', '_')
raw_train = raw_trx.merge(raw_idn, on='TransactionID', how='left')
y_raw = raw_train['isFraud'].copy()
del raw_trx, raw_idn
gc.collect()

class FraudPreprocessorLR(BaseEstimator, TransformerMixin):
    def __init__(self, miss_thresh=0.5):
        self.miss_thresh = miss_thresh
        self.high_miss_cols_ = []
        self.le_store_ = {}
        self.imputer_ = SimpleImputer(strategy='median')
        self.scaler_ = StandardScaler()
        self.final_features = final_features 

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace('-', '_')
        X = self._engineer(X)
        for col in X.select_dtypes(include='object').columns:
            le = LabelEncoder()
            vals = X[col].fillna('unknown').astype(str)
            le.fit(list(vals.unique()) + ['unknown'])
            self.le_store_[col] = le
            X[col] = le.transform(vals)

        X = X[self.final_features]
        X[self.final_features] = self.imputer_.fit_transform(X[self.final_features])
        X[self.final_features] = self.scaler_.fit_transform(X[self.final_features])
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace('-', '_')
        X = self._engineer(X)
        for col, le in self.le_store_.items():
            if col in X.columns:
                known = set(le.classes_)
                X[col] = X[col].fillna('unknown').astype(str).apply(lambda v: v if v in known else 'unknown')
                X[col] = le.transform(X[col])
        
        for f in self.final_features:
            if f not in X.columns: X[f] = 0
            
        X = X[self.final_features]
        X[self.final_features] = self.imputer_.transform(X[self.final_features])
        X[self.final_features] = self.scaler_.transform(X[self.final_features])
        return X

    def _engineer(self, df):
        if 'TransactionDT' in df.columns:
            df['hour'] = ((df['TransactionDT']/3600)%24).astype(np.float32)
            df['dayofweek'] = ((df['TransactionDT']/(3600*24))%7).astype(np.float32)
            df['hour_sin'] = np.sin(2*np.pi*df['hour']/24).astype(np.float32)
            df['hour_cos'] = np.cos(2*np.pi*df['hour']/24).astype(np.float32)
            df['dow_sin'] = np.sin(2*np.pi*df['dayofweek']/7).astype(np.float32)
            df['dow_cos'] = np.cos(2*np.pi*df['dayofweek']/7).astype(np.float32)
        
        for col in ['P_emaildomain', 'R_emaildomain']:
            if col in df.columns:
                df[col+'_suffix'] = df[col].str.split('.').str[-1].fillna('unknown')
                df[col+'_domain'] = df[col].str.split('.').str[0].fillna('unknown')
        
        if 'TransactionAmt' in df.columns:
            df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
            df['TransactionAmt_cents'] = (df['TransactionAmt']%1).astype(np.float32)
            df['amt_is_round'] = (df['TransactionAmt']%1 == 0).astype(np.int8)
            p99 = df['TransactionAmt'].quantile(0.99)
            df['TransactionAmt_capped'] = df['TransactionAmt'].clip(upper=p99)
            
        for g in ['card1', 'addr1']:
            if g in df.columns:
                df[f'{g}_count'] = df.groupby(g)[g].transform('count')
                df[f'{g}_count_log'] = np.log1p(df[f'{g}_count'])
        return df

with mlflow.start_run(run_name=f'{EXP_PREFIX}_Final_Model_Fast'):
    params = rs.best_params_.copy()
    params.update({'max_iter': 100, 'tol': 1e-1})

    mlflow.log_params(params)
    final_clf = LogisticRegression(class_weight='balanced', random_state=42, n_jobs=-1, **params)
    final_pipeline = Pipeline([('preprocessor', FraudPreprocessorLR()), ('classifier', final_clf)])
    
    print("Fitting final model...")
    final_pipeline.fit(raw_train, y_raw)
    mlflow.log_metric("best_cv_score", rs.best_score_)
    sample_input = raw_train.head(5)
    signature = infer_signature(sample_input, final_pipeline.predict_proba(sample_input)[:,1])

    mlflow.sklearn.log_model(
        final_pipeline, 
        artifact_path='lr_pipeline', 
        registered_model_name='LogisticRegression_FraudDetection',
        signature=signature
    )
    print('Registration complete.')

Reloading raw data for final pipeline...
Fitting final model...


2026/05/06 16:46:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:46:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LogisticRegression_FraudDetection' already exists. Creating a new version of this model...
2026/05/06 16:46:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LogisticRegression_FraudDetection, version 3
Created version '3' of model 'LogisticRegression_FraudDetection'.


Registration complete.
🏃 View run LR_Final_Model_Fast at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2/runs/51bde8a72e25456f8a22c9bd416fb34d
🧪 View experiment at: https://dagshub.com/dgrig23/IEEE-CIS-Fraud-Detection---Machine-Learning.mlflow/#/experiments/2
